In [ ]:
import os
import re
import random
import warnings
from pathlib import Path
from functools import lru_cache

import joblib
import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

warnings.filterwarnings("ignore")


class CFG:

    targets = [
        "ACL",
        "MCL",
        "Medial Meniscus",
        "Lateral Meniscus",
        "Medial OA",
        "Lateral OA",
        "PF OA",
        "Effusion",
        "Synovitis",
        "Baker's",
        "Contusion",
        "Fracture",
    ]

    model_dir = Path(
        "/kaggle/input/models/jek1wantaufik/buddy/pytorch/rsna/1"
    )

    output_dir = Path("/kaggle/working")


class IMG:

    size = 192
    triplets = 2

    batch = 3
    workers = 2

    backbone = "resnet18"

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    amp = torch.cuda.is_available()


TARGETS = CFG.targets

PLANES = [
    "Sagittal",
    "Coronal",
    "Axial",
]


def find_root():

    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("work/competition-csv"),
    ]

    if Path("/kaggle/input").exists():
        candidates += list(
            Path("/kaggle/input").glob("*knee*")
        )

    for p in candidates:
        if (p / "test.csv").exists():
            return p

    raise FileNotFoundError(
        "Competition dataset not found."
    )


ROOT = find_root()


MODEL_PATH = CFG.model_dir / "knee_v5.pt"
RIDGE_PATH = CFG.model_dir / "ridge_models.pkl"

assert CFG.model_dir.exists(), (
    f"Model directory not found:\n{CFG.model_dir}"
)

assert MODEL_PATH.exists(), (
    f"Missing model:\n{MODEL_PATH}"
)

assert RIDGE_PATH.exists(), (
    f"Missing ridge models:\n{RIDGE_PATH}"
)

CFG.output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 60)
print("Competition :", ROOT)
print("Model Dir   :", CFG.model_dir)
print("Output Dir  :", CFG.output_dir)
print("Device      :", IMG.device)
print("=" * 60)

In [ ]:
test = pd.read_csv(ROOT / "test.csv")

test_series = pd.read_csv(ROOT / "test_series.csv")

sample = pd.read_csv(ROOT / "sample_submission.csv")


def protocol_features(df, study_ids=None):

    df = df.copy()

    df["slot"] = (
        df.Anatomical_Plane.str[0]
        + "_"
        + df.Fluid_Sensitive.astype(int).astype(str)
    )

    slots = [
        "A_0", "A_1",
        "C_0", "C_1",
        "S_0", "S_1",
    ]

    protocol = pd.crosstab(
        df.StudyInstanceUID,
        df.slot,
    ).reindex(
        columns=slots,
        fill_value=0,
    )

    summary = df.groupby(
        "StudyInstanceUID"
    ).agg(
        n_series=("SeriesInstanceUID", "size"),
        n_planes=("Anatomical_Plane", "nunique"),
        n_fluid=("Fluid_Sensitive", "sum"),
    )

    features = (
        protocol
        .join(summary)
        .astype(np.float32)
    )

    if study_ids is not None:
        features = features.reindex(
            study_ids,
            fill_value=0,
        )

    return features


test_meta = protocol_features(
    test_series,
    test.StudyInstanceUID,
)


assert list(sample.columns) == [
    "StudyInstanceUID",
    *TARGETS,
]

assert len(test) == len(sample)

assert test.StudyInstanceUID.equals(
    sample.StudyInstanceUID
)

assert test_meta.shape[0] == len(test)

print("=" * 60)
print("Test studies :", len(test))
print("Test series  :", len(test_series))
print("Meta shape   :", test_meta.shape)
print("Targets      :", len(TARGETS))
print("=" * 60)

In [ ]:
@lru_cache(maxsize=18000)
def ordered_paths(folder_text):

    folder = Path(folder_text)

    rows = []

    for i, path in enumerate(folder.glob("*.dcm")):

        try:

            ds = pydicom.dcmread(
                path,
                stop_before_pixels=True,
                force=True,
            )

            if (
                hasattr(ds, "ImageOrientationPatient")
                and hasattr(ds, "ImagePositionPatient")
            ):

                orient = np.asarray(
                    ds.ImageOrientationPatient,
                    dtype=float,
                )

                normal = np.cross(
                    orient[:3],
                    orient[3:],
                )

                key = float(
                    np.dot(
                        np.asarray(
                            ds.ImagePositionPatient,
                            dtype=float,
                        ),
                        normal,
                    )
                )

            else:

                key = float(
                    getattr(ds, "InstanceNumber", i)
                )

        except Exception:

            key = float(i)

        rows.append((key, str(path)))

    return tuple(
        p for _, p in sorted(rows)
    )


def decode_triplets(
    folder,
    count=IMG.triplets,
    size=IMG.size,
):

    paths = ordered_paths(str(folder))

    if len(paths) == 0:
        raise FileNotFoundError(folder)

    centers = (
        np.linspace(
            0,
            len(paths) - 1,
            count + 2,
        )[1:-1]
        .round()
        .astype(int)
    )

    images = []

    for center in centers:

        ids = [
            max(0, center - 1),
            center,
            min(len(paths) - 1, center + 1),
        ]

        stack = []

        for idx in ids:

            ds = pydicom.dcmread(
                paths[idx],
                force=True,
            )

            x = ds.pixel_array.astype(np.float32)

            x = (
                x
                * float(getattr(ds, "RescaleSlope", 1.0))
                + float(getattr(ds, "RescaleIntercept", 0.0))
            )

            stack.append(x)

        x = np.stack(stack)

        lo, hi = np.percentile(
            x,
            [0.5, 99.5],
        )

        x = np.clip(
            (x - lo) / (hi - lo + 1e-6),
            0,
            1,
        ).astype(np.float32)

        x = (
            F.interpolate(
                torch.from_numpy(x).unsqueeze(0),
                (size, size),
                mode="bilinear",
                align_corners=False,
            )
            .squeeze(0)
        )

        images.append(x)

    return torch.stack(images)


def choose_series(frame):

    if frame.empty:
        return [None] * len(PLANES)

    selected = []

    for plane in PLANES:

        q = (
            frame[
                frame.Anatomical_Plane.eq(plane)
            ]
            .sort_values(
                [
                    "Fluid_Sensitive",
                    "SeriesInstanceUID",
                ],
                ascending=[False, True],
            )
        )

        selected.append(
            None if q.empty else q.iloc[0]
        )

    return selected


print("✓ DICOM utilities ready")

In [ ]:
class KneeDataset(Dataset):

    def __init__(
        self,
        frame,
        series_frame,
        image_root,
    ):

        self.frame = frame.reset_index(drop=True)

        self.groups = {
            k: v
            for k, v in series_frame.groupby(
                "StudyInstanceUID"
            )
        }

        self.image_root = Path(image_root)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):

        uid = self.frame.iloc[idx].StudyInstanceUID

        images = []
        plane_ids = []
        masks = []

        selected = choose_series(
            self.groups.get(
                uid,
                pd.DataFrame(),
            )
        )

        for plane_id, item in enumerate(selected):

            try:

                folder = (
                    self.image_root
                    / uid
                    / item.SeriesInstanceUID
                )

                x = decode_triplets(folder)

                valid = True

            except Exception:

                x = torch.zeros(
                    IMG.triplets,
                    3,
                    IMG.size,
                    IMG.size,
                    dtype=torch.float32,
                )

                valid = False

            images.append(x)

            plane_ids.extend(
                [plane_id] * IMG.triplets
            )

            masks.extend(
                [valid] * IMG.triplets
            )

        return (
            torch.cat(images, dim=0),
            torch.tensor(
                plane_ids,
                dtype=torch.long,
            ),
            torch.tensor(
                masks,
                dtype=torch.bool,
            ),
            uid,
        )


class KneeNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = timm.create_model(
            IMG.backbone,
            pretrained=False,
            num_classes=0,
            global_pool="avg",
        )

        dim = self.encoder.num_features

        self.plane = nn.Embedding(
            len(PLANES),
            dim,
        )

        self.attn = nn.Sequential(
            nn.Linear(dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.head = nn.Sequential(
            nn.Dropout(0.30),
            nn.Linear(
                dim,
                len(TARGETS),
            ),
        )

    def forward(
        self,
        x,
        plane,
        mask,
    ):

        b, k, c, h, w = x.shape

        feat = self.encoder(
            x.reshape(
                b * k,
                c,
                h,
                w,
            )
        )

        feat = feat.reshape(
            b,
            k,
            -1,
        )

        feat = feat + self.plane(plane)

        attn = (
            self.attn(feat)
            .squeeze(-1)
            .masked_fill(
                ~mask,
                -1e4,
            )
        )

        feat = (
            feat
            * attn.softmax(-1).unsqueeze(-1)
        ).sum(dim=1)

        return self.head(feat)


print("=" * 60)
print("Inference Dataset Ready")
print("Backbone :", IMG.backbone)
print("Image Size :", IMG.size)
print("Triplets :", IMG.triplets)
print("=" * 60)

In [ ]:
ridge_models = joblib.load(RIDGE_PATH)

model = KneeNet().to(IMG.device)

state = torch.load(
    MODEL_PATH,
    map_location=IMG.device,
)

model.load_state_dict(state)

model.eval()

print("✓ Image model loaded")
print("✓ Ridge models loaded")


@torch.no_grad()
def predict_image(
    model,
    flip=True,
):

    dataset = KneeDataset(
        frame=test,
        series_frame=test_series,
        image_root=ROOT / "test_series",
    )

    loader = DataLoader(
        dataset,
        batch_size=IMG.batch,
        shuffle=False,
        num_workers=IMG.workers,
        pin_memory=True,
        persistent_workers=IMG.workers > 0,
    )

    predictions = []
    study_ids = []

    model.eval()

    for x, plane, mask, uid in loader:

        x = x.to(
            IMG.device,
            non_blocking=True,
        )

        plane = plane.to(
            IMG.device,
            non_blocking=True,
        )

        mask = mask.to(
            IMG.device,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=IMG.amp,
        ):

            pred = torch.sigmoid(
                model(
                    x,
                    plane,
                    mask,
                )
            )

            if flip:

                pred_flip = torch.sigmoid(
                    model(
                        torch.flip(
                            x,
                            dims=[-1],
                        ),
                        plane,
                        mask,
                    )
                )

                pred = (
                    pred + pred_flip
                ) * 0.5

        predictions.append(
            pred.cpu().float().numpy()
        )

        study_ids.extend(uid)

    predictions = np.concatenate(
        predictions,
        axis=0,
    )

    assert (
        study_ids
        == test.StudyInstanceUID.tolist()
    )

    return predictions


print("=" * 60)
print("Inference model ready")
print(f"Device : {IMG.device}")
print(f"Flip TTA : Enabled")
print("=" * 60)

In [ ]:
print("=" * 60)
print("Running image inference...")
print("=" * 60)

image_pred = predict_image(
    model=model,
    flip=True,
)

print("Image prediction shape :", image_pred.shape)


Xt = test_meta.values.astype(np.float32)

meta_pred = np.zeros(
    (
        len(test),
        len(TARGETS),
    ),
    dtype=np.float32,
)

for j, ridge in enumerate(ridge_models):

    meta_pred[:, j] = ridge.predict(Xt)

meta_pred = np.clip(
    meta_pred,
    0.02,
    0.98,
)

print("Metadata prediction shape :", meta_pred.shape)


IMAGE_WEIGHT = 0.88
META_WEIGHT = 0.12

final_pred = (
    IMAGE_WEIGHT * image_pred
    + META_WEIGHT * meta_pred
)

final_pred = np.clip(
    final_pred,
    1e-5,
    1 - 1e-5,
)

print("=" * 60)
print("Blend completed")
print("Final prediction shape :", final_pred.shape)
print(
    "Prediction range : "
    f"{final_pred.min():.6f} ~ "
    f"{final_pred.max():.6f}"
)
print("=" * 60)

In [ ]:
def save_submission(
    pred,
    filename="submission.csv",
):

    pred = np.asarray(
        pred,
        dtype=np.float64,
    )

    assert pred.shape == (
        len(test),
        len(TARGETS),
    )

    assert np.isfinite(pred).all()

    submission = sample[
        ["StudyInstanceUID"]
    ].copy()

    submission[TARGETS] = np.clip(
        pred,
        1e-5,
        1 - 1e-5,
    )

    output_path = (
        CFG.output_dir
        / filename
    )

    submission.to_csv(
        output_path,
        index=False,
    )

    print("=" * 60)
    print("Submission created successfully")
    print("=" * 60)
    print(f"Saved to : {output_path}")
    print(f"Shape    : {submission.shape}")
    print(
        "Prediction range : "
        f"{submission[TARGETS].min().min():.6f} ~ "
        f"{submission[TARGETS].max().max():.6f}"
    )
    print("=" * 60)

    return submission


submission = save_submission(
    final_pred,
    filename="submission.csv",
)

display(submission.head())